# Process Recovery

This example notebook demonstrates how to recover a failed process run. This includes:

- Starting a process from a specific node
- Recovering a process while retaining blackboard values from a failed process

<div class="alert alert-info">

**Important**

This notebook requires a running Flowstate solution to connect to. To start a solution:

1. Navigate to [flowstate.intrinsic.ai](https://flowstate.intrinsic.ai/) and sign in
   using your registered Flowstate account.

1. Do **one** of the following:
    - Create a new solution:
        1. Click "Create new solution" and choose "From an example".
        1. Select `pick_and_place:pick_and_place_module2`
        1. Click "Create".
    - Or open an existing solution that was created from the `pick_and_place:pick_and_place_module2` example:
        1. Hover over the solution in the list.
        1. Click "Open solution" or "Start solution".

1. Recommended: Keep the browser tab with the Flowstate solution editor open to watch the effect of notebook actions such as running a skill. You can simultaneously interact with the solution through the web UI and the notebook.

</div>

First, connect to your solution and define convenience shortcuts:

In [ ]:
from intrinsic.solutions import behavior_tree as bt
from intrinsic.solutions import deployments

solution = deployments.connect_to_selected_solution()

executive = solution.executive
skills = solution.skills
world = solution.world

move_robot = skills.ai.intrinsic.move_robot
estimate_pose = skills.ai.intrinsic.estimate_pose

## Simple Recovery

### Inspection process
We define a simple inspection process to illustrate the recovery concepts.
In practice processes requiring recovery are usually significantly larger. We keep the process small as we aim to focus on the concepts.

Our process has three steps:
* Move to a predefined inspection pose
* Inspect the object (we use estimate_pose here)
* Move back to Home

In [ ]:
move_to_inspection_pose = bt.Task(name="Move to Inspection Pose", action=move_robot(
    motion_segments=[
        move_robot.intrinsic_proto.skills.MotionSegment(
            joint_position=world.robot.joint_configurations.view_pose_left,
            motion_type=move_robot.intrinsic_proto.skills.MotionSegment.MotionType.ANY,
        )
    ],
    arm_part=world.robot,
))
def inspect_object(min_num_instances: int):
    return bt.Task(name="Inspect Object", action=estimate_pose(
        camera=solution.resources.wrist_camera,
        pose_estimator=solution.pose_estimators.building_block_ml_estimator,
        return_value_key="estimate_block_result",
        min_num_instances=min_num_instances
    ))
move_home = bt.Task(name="Move Home", action=move_robot(
    motion_segments=[
        move_robot.intrinsic_proto.skills.MotionSegment(
            joint_position=world.robot.joint_configurations.home,
            motion_type=move_robot.intrinsic_proto.skills.MotionSegment.MotionType.ANY,
        )
    ],
    arm_part=world.robot,
))

Now run this process inspecting two objects.

In [ ]:
try:
    executive.run(bt.BehaviorTree(root=bt.Sequence(children=[move_to_inspection_pose, inspect_object(2), move_home])))
except Exception as e:
    print(repr(e))

This will fail as there is only one object. We could now ask the operator to place a second object as expected, or - in our case - fix the process to match reality.

The fixed process only looks for one object.

Instead of starting the process normally, we will start from the `inspect_object` node. This will start the process beginning at the inspection step and skip everything that has already run.

We need to pass a unique identifier for that node in its process tree to `executive.run` in `recover_from_nodes`.
We could define node and tree ids manually and pass them in. Alternatively `get_node_identifier` looks for `fixed_inspect_object` in `fixed_process` and generates these ids automatically.

In [ ]:
fixed_inspect_object = inspect_object(1)
fixed_process = bt.BehaviorTree(root=bt.Sequence(children=[move_to_inspection_pose, fixed_inspect_object, move_home]))
fixed_inspect_object_identifier = fixed_process.get_node_identifier(fixed_inspect_object, autogenerate_missing_ids=True)
executive.run(fixed_process, recover_from_nodes=[fixed_inspect_object_identifier])

## Recovery retaining Blackboard Values

We now perform our inspection multiple times in a loop.

To try failure recovery in this case we artificially inject a failure in the third iteration.

In [ ]:
main_loop = bt.Loop(max_times=5, loop_counter_key="loop_counter")
inspection_step = bt.Branch(if_condition=bt.Blackboard(cel_expression="loop_counter != 2"),
                            then_child=inspect_object(1),
                            else_child=bt.Fail(name="Mock Failure"))
main_loop.do_child = bt.Sequence(children=[move_to_inspection_pose, inspection_step, move_home])
looped_process = bt.BehaviorTree(root=main_loop)

Now run the process and observe a failure in the third iteration.

In [ ]:
try:
    executive.run(looped_process)
except Exception as e:
    print(repr(e))
    print(f"Failed at iteration index {executive.operation.blackboard.get_value('loop_counter')}")

We fix the process by not injecting a failure any more.

In [ ]:
main_loop = bt.Loop(max_times=5, loop_counter_key="loop_counter")
inspection_step = bt.Branch(if_condition=bt.Blackboard(cel_expression="loop_counter != 42"),
                            then_child=inspect_object(1),
                            else_child=bt.Fail(name="Mock Failure"))
main_loop.do_child = bt.Sequence(children=[move_to_inspection_pose, inspection_step, move_home])
looped_process = bt.BehaviorTree(root=main_loop)

We now start at the inspection step. Note that this is not the node that failed (here the `Fail` node).
Recovery can start from any node in a tree.

In [ ]:
inspection_step_identifier = looped_process.get_node_identifier(inspection_step, autogenerate_missing_ids=True)
executive.run(looped_process, recover_from_nodes=[inspection_step_identifier], keep_blackboard=True)

Note that the loop now starts in the third iteration and only performs the missing two iterations.